# R1_X - Exhaustive n-plet discrimination ceiling

This exploratory analysis exhaustively evaluates every n-plet at the configured orders using all scans from all animals **within each dataset**. It ranks raw Omega, mean pairwise Fisher-z, and a pairwise-vector logistic model in both score directions using animal-balanced ROC-AUC. It is a search for the strongest apparent separation in these data, not a held-out or statistically formal estimate of generalization.

For every selected n-plet, the saved files retain all three feature representations and their ROC-AUCs, allowing Omega-selected n-plets to be inspected under the pairwise-vector model. The pairwise-vector probability is fitted and scored on the same scans, so its AUC is especially optimistic. The two pairwise polarities are the model probability and its complement; they are not independent models.

**Runtime note:** pairwise-vector ranking fits one weighted logistic regression for every n-plet. The exhaustive order-4 search can therefore require a substantial amount of time. The search cell reports its candidate/model-fit counts before starting. Run preprocessing 00 first and execute cells from top to bottom.

In [1]:
from pathlib import Path
from itertools import combinations, islice
from math import comb
import heapq
import json
import os
import sys
import warnings

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'src' / 'hoi_anesthesia').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the high-order-anesthesia repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from thoi.measures.gaussian_copula import nplets_measures
from src.hoi_anesthesia.io import load_covariance_dict
from src.hoi_anesthesia.utils import pairwise_fisher_z
from src.hoi_anesthesia.validation import load_classification_data

print(f'Repository root: {ROOT}')
print(f'Available compute device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

Repository root: c:\CAMILO\Brain_Entropy\CODES\high-order-anesthesia
Available compute device: CUDA


c:\CAMILO\venvs\.high-order-anesthesia_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Search configuration

MA and DBS remain separate because they are distinct datasets; within each one, every eligible scan from every animal is pooled for the exploratory score. Responsive scans are the positive class. Animal-balanced weights prevent animals with more scans from dominating the ROC-AUC. Change `ORDERS` to include order 5 later, or change `TOP_K` to retain a different number of candidates.

In [2]:
DATASETS = ('MA', 'DBS')
ORDERS = (3, 4, 5)
TOP_K = 100
BATCH_SIZE = 14000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
COVARIANCE_PATH = ROOT / 'data' / 'covariance_matrices_gc.h5'
METADATA_PATH = ROOT / 'data' / 'scan_metadata.csv'
OUTPUT = ROOT / 'results' / 'R1_X_exhaustive_maximum'

if not ORDERS or any(order < 3 for order in ORDERS):
    raise ValueError('ORDERS must contain integers of at least 3.')
if TOP_K < 1 or BATCH_SIZE < 1:
    raise ValueError('TOP_K and BATCH_SIZE must be positive.')

print(f'Datasets: {DATASETS}')
print(f'Orders: {ORDERS}; top K per measure and polarity: {TOP_K}')
print(f'Pairwise-vector model: weighted logistic regression, C=1, same-scan fit and score')
print(f'Output directory: {OUTPUT}')

Datasets: ('MA', 'DBS')
Orders: (3, 4, 5); top K per measure and polarity: 100
Pairwise-vector model: weighted logistic regression, C=1, same-scan fit and score
Output directory: c:\CAMILO\Brain_Entropy\CODES\high-order-anesthesia\results\R1_X_exhaustive_maximum


## Load and align all scans

The existing metadata alignment helper defines responsive labels consistently with R1_A. Each class receives total weight 1, divided equally among its animals and then among scans within each animal. This changes the ROC-AUC weighting, not the set of scans used.

In [3]:
metadata_all = pd.read_csv(METADATA_PATH, dtype={'animal': str, 'session': str, 'run': str})
all_covariances = load_covariance_dict(COVARIANCE_PATH)
DATA = {}

for dataset in DATASETS:
    covariance_array, scan_rows = load_classification_data(
        all_covariances, metadata_all, dataset,
    )
    scan_rows = scan_rows.reset_index(drop=True)
    labels = scan_rows['responsive'].to_numpy(dtype=int)
    animals = scan_rows['animal'].astype(str).to_numpy()
    weights = np.zeros(len(scan_rows), dtype=float)
    for class_value in (0, 1):
        class_mask = labels == class_value
        class_animals, scan_counts = np.unique(animals[class_mask], return_counts=True)
        for animal, scan_count in zip(class_animals, scan_counts):
            weights[class_mask & (animals == animal)] = 1.0 / (len(class_animals) * scan_count)
    if set(np.unique(labels)) != {0, 1} or not np.isclose(weights[labels == 0].sum(), 1.0):
        raise ValueError(f'{dataset}: both response classes and valid animal weights are required.')
    if not np.isclose(weights[labels == 1].sum(), 1.0):
        raise ValueError(f'{dataset}: responsive animal weights do not sum to one.')
    DATA[dataset] = {
        'covariances': covariance_array,
        'scans': scan_rows,
        'labels': labels,
        'weights': weights,
    }
    print(
        f'{dataset}: covariances={covariance_array.shape}; scans={len(scan_rows)}; '
        f'animals={scan_rows["animal"].nunique()}; regions={covariance_array.shape[1]}; '
        f'responsive={int(labels.sum())}; non-responsive={int((labels == 0).sum())}; '
        f'class weight sums=({weights[labels == 0].sum():.3f}, {weights[labels == 1].sum():.3f})'
    )

MA: covariances=(80, 82, 82); scans=80; animals=5; regions=82; responsive=24; non-responsive=56; class weight sums=(1.000, 1.000)
DBS: covariances=(100, 82, 82); scans=100; animals=5; regions=82; responsive=36; non-responsive=64; class weight sums=(1.000, 1.000)


## Estimate exhaustive workload

There are six rankings per order and dataset: three representations times two score directions. Pairwise-vector fitting is performed once per candidate; its reversed-probability ranking is derived from the same fitted model.

In [4]:
WORKLOAD = []
for dataset in DATASETS:
    n_regions = DATA[dataset]['covariances'].shape[1]
    for order in ORDERS:
        if order > n_regions:
            raise ValueError(f'{dataset}: order {order} exceeds the {n_regions} available regions.')
        n_candidates = comb(n_regions, order)
        WORKLOAD.append({
            'dataset': dataset,
            'order': order,
            'regions': n_regions,
            'candidates': n_candidates,
            'pairwise_logistic_fits': n_candidates,
            'ranked_candidates': 6 * TOP_K,
        })
WORKLOAD = pd.DataFrame(WORKLOAD)
display(WORKLOAD)
print(f'Total exhaustive candidates / pairwise model fits: {WORKLOAD["candidates"].sum():,}')
print('The pairwise-vector search is intentionally exhaustive and may take a long time.')

,dataset,order,regions,candidates,pairwise_logistic_fits,ranked_candidates
0,MA,3,82,88560,88560,600
1,MA,4,82,1749060,1749060,600
2,MA,5,82,27285336,27285336,600
3,DBS,3,82,88560,88560,600
4,DBS,4,82,1749060,1749060,600
5,DBS,5,82,27285336,27285336,600


Total exhaustive candidates / pairwise model fits: 58,245,912
The pairwise-vector search is intentionally exhaustive and may take a long time.


## Exhaustive scoring and top-K retention

For Omega and mean Fisher-z, ROC-AUC is computed directly from the scalar scan scores. For pairwise vectors, a weighted logistic model (the same fixed L2/C=1 family used by the existing feature evaluation) is fitted separately for each n-plet and scored on those same scans. Ties in the retained top-K are resolved by lexicographically smaller region-index tuples. No fold or held-out animal is used in this exploratory search.

In [ ]:
TOP_HEAPS = {}
SEARCH_LOG = []

for dataset in DATASETS:
    covariance_tensor = torch.as_tensor(
        DATA[dataset]['covariances'], dtype=torch.float64, device=DEVICE,
    )
    labels = DATA[dataset]['labels']
    weights = DATA[dataset]['weights']
    n_regions = covariance_tensor.shape[-1]

    for order in ORDERS:
        candidate_iterator = combinations(range(n_regions), order)
        candidate_batch = list(islice(candidate_iterator, BATCH_SIZE))
        processed = 0
        n_candidates = comb(n_regions, order)
        print(f'Starting {dataset}, order {order}: {n_candidates:,} candidates')

        while candidate_batch:
            candidate_tensor = torch.as_tensor(
                candidate_batch, dtype=torch.long, device=DEVICE,
            )
            hoi_batch = nplets_measures(
                covariance_tensor,
                nplets=candidate_tensor,
                covmat_precomputed=True,
                T=None,
                device=DEVICE,
                batch_size=BATCH_SIZE,
                verbose=40,
            )
            omega_batch = hoi_batch[..., 2].detach().cpu().numpy()
            pairwise_batch = pairwise_fisher_z(
                covariance_tensor, candidate_tensor,
            ).detach().cpu().numpy()
            mean_fisher_z_batch = pairwise_batch.mean(axis=-1)

            for candidate_position, candidate in enumerate(candidate_batch):
                candidate_tuple = tuple(int(region) for region in candidate)
                scalar_scores = {
                    'omega': omega_batch[candidate_position],
                    'mean_fisher_z': mean_fisher_z_batch[candidate_position],
                }
                aucs = {}
                for measure_name, scan_scores in scalar_scores.items():
                    for polarity in (1, -1):
                        aucs[(measure_name, polarity)] = roc_auc_score(
                            labels, polarity * scan_scores, sample_weight=weights,
                        )

                pairwise_features = pairwise_batch[candidate_position]
                pairwise_scaler = StandardScaler().fit(
                    pairwise_features, sample_weight=weights,
                )
                pairwise_scaled = pairwise_scaler.transform(pairwise_features)
                pairwise_model = LogisticRegression(
                    C=1, penalty='l2', solver='lbfgs', max_iter=10000,
                )
                with warnings.catch_warnings(record=True) as fit_warnings:
                    warnings.simplefilter('always', ConvergenceWarning)
                    pairwise_model.fit(pairwise_scaled, labels, sample_weight=weights)
                if pairwise_model.n_iter_[0] >= pairwise_model.max_iter or any(
                    issubclass(item.category, ConvergenceWarning) for item in fit_warnings
                ):
                    raise RuntimeError(
                        f'Pairwise logistic model did not converge: {dataset}, '
                        f'order {order}, n-plet {candidate_tuple}.'
                    )
                pairwise_probability = pairwise_model.predict_proba(pairwise_scaled)[:, 1]
                aucs[('pairwise_vector', 1)] = roc_auc_score(
                    labels, pairwise_probability, sample_weight=weights,
                )
                aucs[('pairwise_vector', -1)] = roc_auc_score(
                    labels, 1.0 - pairwise_probability, sample_weight=weights,
                )

                for (measure_name, polarity), auc in aucs.items():
                    heap_key = (dataset, order, measure_name, polarity)
                    if heap_key not in TOP_HEAPS:
                        TOP_HEAPS[heap_key] = []
                    heap_entry = (
                        float(auc),
                        tuple(-region for region in candidate_tuple),
                        candidate_tuple,
                    )
                    if len(TOP_HEAPS[heap_key]) < TOP_K:
                        heapq.heappush(TOP_HEAPS[heap_key], heap_entry)
                    elif heap_entry[:2] > TOP_HEAPS[heap_key][0][:2]:
                        heapq.heapreplace(TOP_HEAPS[heap_key], heap_entry)

            processed += len(candidate_batch)
            if processed == n_candidates or processed % (BATCH_SIZE * 10) == 0:
                progress = {
                    'dataset': dataset,
                    'order': order,
                    'processed': processed,
                    'total': n_candidates,
                    'fraction_complete': processed / n_candidates,
                }
                SEARCH_LOG.append(progress)
                print(f'{dataset} order {order}: {processed:,}/{n_candidates:,} candidates')
            candidate_batch = list(islice(candidate_iterator, BATCH_SIZE))

print('Exhaustive search finished for every configured dataset and order.')

Starting MA, order 3: 88,560 candidates


## Inspect the six top-K rankings

Each dataset/order has separate top-K lists for Omega, mean Fisher-z, and the pairwise-vector probability, each in both polarities. A candidate may occur in multiple lists; its list memberships are preserved in the saved ranking table.

In [ ]:
TOP_ROWS = []
for (dataset, order, measure_name, polarity), candidate_heap in TOP_HEAPS.items():
    ordered_candidates = sorted(candidate_heap, key=lambda entry: (entry[0], entry[1]), reverse=True)
    for rank, (auc, _, candidate_tuple) in enumerate(ordered_candidates, start=1):
        TOP_ROWS.append({
            'dataset': dataset,
            'order': order,
            'measure': measure_name,
            'polarity': polarity,
            'rank': rank,
            'roc_auc': auc,
            'nplet': json.dumps(candidate_tuple),
        })
TOP_TABLE = pd.DataFrame(TOP_ROWS).sort_values(
    ['dataset', 'order', 'measure', 'polarity', 'rank'],
    ignore_index=True,
)

ranking_counts = TOP_TABLE.groupby(
    ['dataset', 'order', 'measure', 'polarity'], as_index=False,
).agg(n_retained=('nplet', 'size'), best_roc_auc=('roc_auc', 'max'))
display(ranking_counts)
display(TOP_TABLE.groupby(
    ['dataset', 'order', 'measure', 'polarity'], as_index=False,
).head(10))

## Re-evaluate the selected union and retain per-scan values

For the union of selected n-plets, this cell calculates all three representations and both ROC-AUC directions, regardless of which representation selected the n-plet. It also stores the fitted pairwise-model probability, the full pairwise vector, scan metadata, and the ranking memberships. This is the comparison table for asking how Omega-selected candidates behave under pairwise evaluation.

In [ ]:
SUMMARY_ROWS = []
SCAN_VALUE_ROWS = []

for dataset in DATASETS:
    covariance_tensor = torch.as_tensor(
        DATA[dataset]['covariances'], dtype=torch.float64, device=DEVICE,
    )
    scans = DATA[dataset]['scans']
    labels = DATA[dataset]['labels']
    weights = DATA[dataset]['weights']

    for order in ORDERS:
        order_rows = TOP_TABLE.loc[
            (TOP_TABLE['dataset'] == dataset) & (TOP_TABLE['order'] == order)
        ]
        selected_nplets = sorted({tuple(json.loads(value)) for value in order_rows['nplet']})

        for candidate_tuple in selected_nplets:
            candidate_tensor = torch.as_tensor(
                [candidate_tuple], dtype=torch.long, device=DEVICE,
            )
            hoi_values = nplets_measures(
                covariance_tensor,
                nplets=candidate_tensor,
                covmat_precomputed=True,
                T=None,
                device=DEVICE,
                batch_size=BATCH_SIZE,
                verbose=40,
            )[0].detach().cpu().numpy()
            omega_values = hoi_values[:, 2]
            pairwise_values = pairwise_fisher_z(
                covariance_tensor, candidate_tensor,
            )[0].detach().cpu().numpy()
            mean_fisher_z_values = pairwise_values.mean(axis=-1)

            pairwise_scaler = StandardScaler().fit(pairwise_values, sample_weight=weights)
            pairwise_scaled = pairwise_scaler.transform(pairwise_values)
            pairwise_model = LogisticRegression(
                C=1, penalty='l2', solver='lbfgs', max_iter=10000,
            )
            pairwise_model.fit(pairwise_scaled, labels, sample_weight=weights)
            if pairwise_model.n_iter_[0] >= pairwise_model.max_iter:
                raise RuntimeError(
                    f'Pairwise logistic model did not converge for {dataset}, '
                    f'order {order}, n-plet {candidate_tuple}.'
                )
            pairwise_probability = pairwise_model.predict_proba(pairwise_scaled)[:, 1]

            candidate_auc = {}
            for measure_name, scan_scores in (
                ('omega', omega_values),
                ('mean_fisher_z', mean_fisher_z_values),
                ('pairwise_vector', pairwise_probability),
            ):
                for polarity in (1, -1):
                    directed_scores = scan_scores if polarity == 1 else 1.0 - scan_scores
                    if measure_name != 'pairwise_vector' and polarity == -1:
                        directed_scores = -scan_scores
                    candidate_auc[f'{measure_name}_auc_polarity_{polarity:+d}'] = roc_auc_score(
                        labels, directed_scores, sample_weight=weights,
                    )

            candidate_id = f'{dataset}_n{order}_' + '_'.join(map(str, candidate_tuple))
            memberships = []
            candidate_memberships = order_rows.loc[
                order_rows['nplet'].eq(json.dumps(candidate_tuple))
            ]
            for _, membership_row in candidate_memberships.iterrows():
                memberships.append(
                    f'{membership_row["measure"]}:{membership_row["polarity"]:+d}:'
                    f'rank{membership_row["rank"]}'
                )
            SUMMARY_ROWS.append({
                'dataset': dataset,
                'order': order,
                'candidate_id': candidate_id,
                'nplet': json.dumps(candidate_tuple),
                'selection_memberships': json.dumps(memberships),
                'selected_by_omega': any(item.startswith('omega:') for item in memberships),
                'selected_by_pairwise_vector': any(
                    item.startswith('pairwise_vector:') for item in memberships
                ),
                **candidate_auc,
            })

            for scan_position, scan_row in enumerate(scans.to_dict(orient='records')):
                SCAN_VALUE_ROWS.append({
                    'dataset': dataset,
                    'order': order,
                    'candidate_id': candidate_id,
                    'nplet': json.dumps(candidate_tuple),
                    'selection_memberships': json.dumps(memberships),
                    'responsive': int(labels[scan_position]),
                    'omega': float(omega_values[scan_position]),
                    'mean_fisher_z': float(mean_fisher_z_values[scan_position]),
                    'pairwise_vector': json.dumps(pairwise_values[scan_position].tolist()),
                    'pairwise_probability_responsive': float(pairwise_probability[scan_position]),
                    **{key: value for key, value in scan_row.items()
                       if key not in {'responsive', 'dataset'}},
                })

SELECTED_SUMMARY = pd.DataFrame(SUMMARY_ROWS)
SELECTED_SCAN_VALUES = pd.DataFrame(SCAN_VALUE_ROWS)
display(SELECTED_SUMMARY.groupby(
    ['dataset', 'order', 'selected_by_omega', 'selected_by_pairwise_vector'],
    as_index=False,
).agg(n_unique_candidates=('candidate_id', 'nunique')))
display(SELECTED_SUMMARY.head())
display(SELECTED_SCAN_VALUES.head())

## Save rankings and retained measurements

The ranking file has six top-K lists per order and dataset. The candidate summary stores all six ROC-AUC directions for each unique selected n-plet. The scan-level file stores raw Omega, mean Fisher-z, the JSON-encoded pairwise vector, and pairwise-model probability for every scan.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
for dataset in DATASETS:
    TOP_TABLE.loc[TOP_TABLE['dataset'] == dataset].to_csv(
        OUTPUT / f'{dataset}_top_candidates.csv', index=False,
    )
    SELECTED_SUMMARY.loc[SELECTED_SUMMARY['dataset'] == dataset].to_csv(
        OUTPUT / f'{dataset}_selected_candidate_summary.csv', index=False,
    )
    SELECTED_SCAN_VALUES.loc[SELECTED_SCAN_VALUES['dataset'] == dataset].to_csv(
        OUTPUT / f'{dataset}_selected_scan_values.csv', index=False,
    )
print(f'Saved per-dataset rankings, candidate summaries, and scan-level values under {OUTPUT}')

## Visualize the apparent search ceiling

The first plot shows the ROC-AUC distributions among the retained top-K candidates for each search track. The second compares the per-state mean Omega for candidates selected by Omega, as a descriptive view of the awake-synergy hypothesis. These plots describe selected candidates only and do not provide inferential tests.

In [ ]:
for dataset in DATASETS:
    dataset_top = TOP_TABLE.loc[TOP_TABLE['dataset'] == dataset].copy()
    dataset_top['search_track'] = (
        dataset_top['measure'] + ' / ' + dataset_top['polarity'].map({1: '+', -1: '-'})
    )
    tracks = list(dataset_top['search_track'].drop_duplicates())
    figure, axis = plt.subplots(figsize=(12, 5))
    box_values = [dataset_top.loc[dataset_top['search_track'] == track, 'roc_auc'] for track in tracks]
    axis.boxplot(box_values, labels=tracks, showfliers=False)
    axis.axhline(0.5, color='black', linestyle='--', linewidth=1)
    axis.set_title(f'{dataset}: ROC-AUC among retained top-K candidates')
    axis.set_ylabel('Animal-balanced ROC-AUC (apparent, all scans)')
    axis.tick_params(axis='x', rotation=25)
    figure.tight_layout()
    plt.show()

    omega_candidate_ids = SELECTED_SUMMARY.loc[
        (SELECTED_SUMMARY['dataset'] == dataset)
        & SELECTED_SUMMARY['selected_by_omega'], 'candidate_id',
    ]
    omega_scan_values = SELECTED_SCAN_VALUES.loc[
        (SELECTED_SCAN_VALUES['dataset'] == dataset)
        & SELECTED_SCAN_VALUES['candidate_id'].isin(omega_candidate_ids)
    ]
    state_omega = omega_scan_values.groupby(
        ['order', 'state'], as_index=False,
    ).agg(mean_omega=('omega', 'mean'), n_scan_values=('omega', 'size'))
    display(state_omega)
    figure, axis = plt.subplots(figsize=(10, 5))
    for order in ORDERS:
        order_values = state_omega.loc[state_omega['order'] == order]
        axis.plot(
            order_values['state'], order_values['mean_omega'],
            marker='o', label=f'order {order}',
        )
    axis.axhline(0, color='black', linewidth=1)
    axis.set_title(f'{dataset}: Omega-selected candidates by state')
    axis.set_ylabel('Mean raw Omega across selected candidates and scans')
    axis.tick_params(axis='x', rotation=25)
    axis.legend()
    figure.tight_layout()
    plt.show()

## Interpretation limits

- The top AUCs are maxima chosen after searching many candidates and are scored on the same scans used to choose them. They quantify an apparent ceiling in these data, not expected out-of-sample performance.
- Pairwise-vector probabilities are in-sample fits. Their AUCs are descriptive and should not be treated as directly comparable estimates of generalization against raw scalar Omega or mean Fisher-z.
- The pairwise polarity rankings are exact probability reversals, not separate fitted models. For a fitted probability without ties, their ROC-AUCs sum to one.
- The saved top-K lists can overlap. Use `selection_memberships` and `candidate_id` to compare the same n-plet across selection tracks and feature representations.
- The state plot describes raw Omega among selected candidates. It does not test the awake-synergy hypothesis statistically.